In [1]:
import scarf

scarf.fetch_dataset(
    "kang_15K_pbmc_rnaseq",
    save_path="scarf_datasets",
    as_zarr=True,
)
ds = scarf.DataStore(
    "scarf_datasets/kang_15K_pbmc_rnaseq/data.zarr",
    nthreads=4,
)
ds

DataStore has 8487 (14619) cells with 1 assays: RNA
   Cell metadata:
            'I', 'ids', 'names', 'RNA_percentMito', 'cluster_labels', 
            'RNA_percentRibo', 'RNA_leiden_cluster', 'RNA_nFeatures', 'RNA_nCounts', 'RNA_UMAP1', 
            'RNA_UMAP2'
   RNA assay has 11352 (35635) features and following metadata:
            'I', 'ids', 'names', 'I__hvgs', 'nCells', 
            'dropOuts'

In [2]:
# Ensure a clustering column exists for aggregation demos.
if "RNA_leiden_cluster" not in ds.cells.columns:
    ds.mark_hvgs(min_cells=20, top_n=500)
    ds.make_graph(feat_key="hvgs", k=11, dims=15, n_centroids=100)
    ds.run_leiden_clustering(resolution=0.5)

In [3]:
ds.run_marker_search(group_key="RNA_leiden_cluster", gene_batch_size=100)
markers = ds.get_markers(
    group_key="RNA_leiden_cluster",
    group_id="1",
    min_score=-1,
    min_frac_exp=-1,
)
markers.head()

Saved marker results to RNA/markers/I__RNA_leiden_cluster in 0.9s (24 clusters, layout=compact_v2)



,group_id,feature_name,score,mean,mean_rest,frac_exp,frac_exp_rest,fold_change,p_value,feature_index
0,1,RP11-1399P15.1,0.18551,0.08003,0.01006,0.09083,0.01514,7.95448,2.822936e-51,1468
1,1,GPR171,0.16657,0.16902,0.03238,0.19358,0.04623,5.22058,2.470904e-77,2402
2,1,SESN3,0.15856,0.22707,0.05136,0.23303,0.07030,4.42138,1.741056e-71,6676
3,1,PERP,0.15463,0.08077,0.01627,0.09633,0.02433,4.96378,1.377799e-35,3954
4,1,TMEM173,0.15302,0.12888,0.02361,0.15046,0.03542,5.45904,5.381645e-61,3268


In [4]:
bulk = ds.make_bulk(
    group_key="RNA_leiden_cluster",
    aggr_type="sum",
    feature_label="name",
)
bulk.iloc[:5, :5]

,1,2,3,4,5
RP11-34P13.8,0,0,0,0,0
AL627309.1,1,0,1,0,0
RP11-206L10.2,0,1,0,0,1
RP11-206L10.9,0,0,0,0,1
FAM87B,0,0,0,0,1


In [5]:
bulk_reps = ds.make_bulk(
    group_key="RNA_leiden_cluster",
    aggr_type="sum",
    feature_label="name",
    pseudo_reps=2,
)
bulk_reps.shape

(16910, 48)

In [6]:
bulk.to_csv("scarf_datasets/kang_pseudobulk_counts.csv")